<a href="https://colab.research.google.com/github/ayanguin/NLP-Poster-Diagnosing-Evaluation-Instability-via-Deep-Linguistic-Fingerprinting/blob/main/FinalScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section - Initialization

In [1]:
pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.9/312.9 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.2/806.2 kB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 10

In [1]:
!pip uninstall -y torchaudio torchvision

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0


In [2]:
!pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 74.2 MB/s eta 0:00:00


# Environment setup

In [3]:
import sys
import os
import io
import re
import polars as pl
from datasets import load_dataset
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

from google.colab import drive
from google.colab import files

# Data Selection and Pre-Processing

In [4]:
MMLU_DATASET = load_dataset("cais/mmlu", "all", split="test")
TRUTHFULQA_DATASET = load_dataset("truthfulqa/truthful_qa", "multiple_choice", split="validation")
OPINIONQA_DATASET = load_dataset("timchen0618/opinionqa", split="test")

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.50MB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  408kB            

...

README.md:   0%|          | 0.00/773 [00:00<?, ?B/s]

opinionqa.dev.jsonl:   0%|          | 0.00/95.9k [00:00<?, ?B/s]

opinionqa.test.jsonl:   0%|          | 0.00/313k [00:00<?, ?B/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [5]:
from datasets import concatenate_datasets

# 1. Define standardization functions for each dataset's unique structure
def map_mmlu(row):
    return {
        "Standard_Question": row["question"],
        "Standard_Options": row["choices"],
        "Dataset_Source": "MMLU"
    }

def map_opinionqa(row):
    return {
        "Standard_Question": row["question"],
        "Standard_Options": row["perspectives"],
        "Dataset_Source": "OpinionQA"
    }

def map_truthfulqa(row):
    # TruthfulQA nests its multiple-choice options inside the 'mc1_targets' dictionary
    return {
        "Standard_Question": row["question"],
        "Standard_Options": row["mc1_targets"]["choices"],
        "Dataset_Source": "TruthfulQA"
    }

# 2. Apply the mapping and isolate ONLY the new standard columns
cols_to_keep = ["Standard_Question", "Standard_Options", "Dataset_Source"]

mmlu_clean = MMLU_DATASET.map(map_mmlu).select_columns(cols_to_keep)
opinionqa_clean = OPINIONQA_DATASET.map(map_opinionqa).select_columns(cols_to_keep)
truthfulqa_clean = TRUTHFULQA_DATASET.map(map_truthfulqa).select_columns(cols_to_keep)

# 3. Combine them into one unified master dataset
master_dataset = concatenate_datasets([mmlu_clean, opinionqa_clean, truthfulqa_clean])

# Shuffle to ensure the model doesn't just see one benchmark at a time
master_dataset = master_dataset.shuffle(seed=42)

print(f"Combined Master Dataset successfully created!")
print(f"Total rows: {len(master_dataset)}")
print(f"Unified Columns: {master_dataset.column_names}")

Map:   0%|          | 0/14042 [00:00<?, ? examples/s]

Map:   0%|          | 0/882 [00:00<?, ? examples/s]

Map:   0%|          | 0/817 [00:00<?, ? examples/s]

Combined Master Dataset successfully created!
Total rows: 15741
Unified Columns: ['Standard_Question', 'Standard_Options', 'Dataset_Source']


In [ ]:
print(master_dataset)
print(mmlu_clean)
print(opinionqa_clean)
print(truthfulqa_clean)

In [6]:
import polars as pl

# 1. Grab the first 3 rows of the Hugging Face dataset (returns a dictionary)
sample_data = master_dataset[:3]

# 2. Convert to Polars for pretty formatting
preview_df = pl.DataFrame(sample_data)

# 3. Expand the table width so the options list doesn't get cut off with '...'
pl.Config.set_tbl_width_chars(150)

print("=== MASTER DATASET PREVIEW ===")
print(preview_df)

=== MASTER DATASET PREVIEW ===
shape: (3, 3)
┌─────────────────────────────────┬─────────────────────────────────┬────────────────┐
│ Standard_Question               ┆ Standard_Options                ┆ Dataset_Source │
│ ---                             ┆ ---                             ┆ ---            │
│ str                             ┆ list[str]                       ┆ str            │
╞═════════════════════════════════╪═════════════════════════════════╪════════════════╡
│ Which of the following is not … ┆ ["As a planet moves around its… ┆ MMLU           │
│ Name three of the five main us… ┆ ["Touch, feel, stroke.", "Grip… ┆ MMLU           │
│ Pumice is a volcanic rock that… ┆ ["less.", "equal.", … "none be… ┆ MMLU           │
└─────────────────────────────────┴─────────────────────────────────┴────────────────┘


In [7]:
# Data set filter (total 900: 300 from each dataset)

from datasets import concatenate_datasets

print("Filtering and sampling 300 rows from each source...")

# 1. Filter the master dataset by the 'Dataset_Source' column we created earlier
mmlu_full = master_dataset.filter(lambda x: x["Dataset_Source"] == "MMLU")
opinionqa_full = master_dataset.filter(lambda x: x["Dataset_Source"] == "OpinionQA")
truthfulqa_full = master_dataset.filter(lambda x: x["Dataset_Source"] == "TruthfulQA")

# 2. Shuffle each dataset with a fixed seed (42) for reproducibility, then select 300 rows
sample_mmlu = mmlu_full.shuffle(seed=42).select(range(300))
sample_opinion = opinionqa_full.shuffle(seed=42).select(range(300))
sample_truthful = truthfulqa_full.shuffle(seed=42).select(range(300))

# 3. Combine them into a new balanced master dataset
balanced_master_dataset = concatenate_datasets([sample_mmlu, sample_opinion, sample_truthful])

# 4. Shuffle the combined dataset
# This ensures the model doesn't answer 300 MMLU questions in a row before seeing an OpinionQA question
balanced_master_dataset = balanced_master_dataset.shuffle(seed=42)

# 5. Verify the final shape
print("Balanced master dataset successfully created!")
print(f"Total rows: {len(balanced_master_dataset)}")
print(balanced_master_dataset)

Filtering and sampling 300 rows from each source...


Filter:   0%|          | 0/15741 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15741 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15741 [00:00<?, ? examples/s]

Balanced master dataset successfully created!
Total rows: 900
Dataset({
    features: ['Standard_Question', 'Standard_Options', 'Dataset_Source'],
    num_rows: 900
})


# Model Selection and Setup

In [8]:
QWEN = "Qwen/Qwen2.5-1.5B-Instruct"           # Need to run T4
META = "meta-llama/Llama-3.2-3B-Instruct"     # Need to run T4
HF = "HuggingFaceTB/SmolLM2-1.7B-Instruct"    # Need to run T4
GOOGLE = "google/gemma-2-2b-it"               # Need to run A100 GPU
MISTRAL = "mistralai/Mistral-7B-Instruct-v0.3"     # Need to run A100 GPU

In [9]:
import ipywidgets as widgets

from IPython.display import display

dropdown = widgets.Dropdown(
    options=[
        ("QWEN", QWEN),
        ("META", META),
        ("HuggingFace", HF),
        ("GOOGLE", GOOGLE),
        ("MISTRAL", MISTRAL)
    ],
    description="Model:"
)

display(dropdown)


Dropdown(description='Model:', options=(('QWEN', 'Qwen/Qwen2.5-1.5B-Instruct'), ('META', 'meta-llama/Llama-3.2…

In [10]:
# TRIGGER THIS WHEN WANT TO RUN MODEL Llama and Google model
from google.colab import userdata
from huggingface_hub import login

# Securely fetch the token from Colab Secrets
hf_token = userdata.get('HF_Token')
login(token=hf_token)

In [11]:
selected_option = dropdown.value

class FilenoFix:
    """Wraps a stream and exposes a real OS-level fileno(),
    working around ipykernel's OutStream not supporting it."""
    def __init__(self, stream, fd):
        self._stream = stream
        self._fd = fd
    def __getattr__(self, name):
        return getattr(self._stream, name)
    def fileno(self):
        return self._fd
    def writable(self):
        return True

"""Applies the fileno() patch only if it is actually broken."""
try:
    sys.stdout.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stdout = FilenoFix(sys.stdout, 1)

try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stderr = FilenoFix(sys.stderr, 2)

# 3. Load and return the model
print(f"Loading {selected_option} into GPU memory...")
if selected_option == "google/gemma-2-2b-it":
  llm = LLM(
      model=selected_option,
      dtype="bfloat16",
      max_model_len=2048,
      enforce_eager=True
  )
else:
  llm = LLM(
      model=selected_option,
      dtype="half",
      max_model_len=2048,
      enforce_eager=True
  )


Loading HuggingFaceTB/SmolLM2-1.7B-Instruct into GPU memory...
INFO 08-21 08:39:59 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'HuggingFaceTB/SmolLM2-1.7B-Instruct'}


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

INFO 08-21 08:40:21 [model.py:645] Resolved architecture: LlamaForCausalLM
WARNING 08-21 08:40:21 [model.py:2217] Casting torch.bfloat16 to torch.float16.
INFO 08-21 08:40:21 [model.py:1883] Using max model len 2048
INFO 08-21 08:40:21 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 08-21 08:40:21 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-21 08:40:21 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 08-21 08:40:21 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 08-21 08:40:21 [vllm.py:1426] Cudagraph is disabled under eager mode
INFO 08-21 08:40:21 [compilation.py:329] Enabled custom fusions: norm_quant, act_quan

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

INFO 08-21 08:40:29 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='HuggingFaceTB/SmolLM2-1.7B-Instruct', speculative_config=None, tokenizer='HuggingFaceTB/SmolLM2-1.7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_en

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 08-21 08:41:31 [default_loader.py:430] Loading weights took 13.76 seconds
INFO 08-21 08:41:32 [model_runner.py:329] Model loading took 3.19 GiB and 58.494674 seconds
WARNING 08-21 08:41:32 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
INFO 08-21 08:41:38 [gpu_worker.py:563] Available KV cache memory: 9.58 GiB
INFO 08-21 08:41:39 [kv_cache_utils.py:2235] GPU KV cache size: 52,288 tokens
INFO 08-21 08:41:39 [kv_cache_utils.py:2236] Maximum concurrency for 2,048 tokens per request: 25.53x
INFO 08-21 08:41:39 [gpu_worker.py:789] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.92, 13.4 GiB). Actual usage is 3.35 GiB for consumed memory (weights + non-torch), 0.47 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=10124701840` (9.43 GiB) to fit into requested

# Data 2 Model

In [12]:



# ==========================================
# PHASE 1 & 2: DATA PREP & MODEL INIT
# ==========================================
print("Loading model and tokenizer...")
#MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# 1. Load the tokenizer to handle the chat templates
tokenizer = AutoTokenizer.from_pretrained(selected_option)     #NEED TO CHANGE THE MODEL NAME EVERY TIME BEFORE RUNNING THE CODE


print("Loading and standardizing dataset...")
#dataset = load_dataset("timchen0618/opinionqa", split="test")
dataset = balanced_master_dataset

def standardize_and_prompt(row):
    question = row.get("Standard_Question")
    options = row.get("Standard_Options")

    labels = ["A", "B", "C", "D", "E", "F"]
    formatted_options = ""
    for i, option in enumerate(options):
        if i < len(labels):
            formatted_options += f"{labels[i]}) {option}\n"

    # Base raw text
    raw_constrained = (
        f"Answer the following multiple-choice question by outputting ONLY "
        f"the single letter (A, B, C, etc.) corresponding to the correct option.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{formatted_options}\nAnswer:"
    )

    raw_unconstrained = (
        f"Read the following question and the provided options. "
        f"Take a clear stance, explain your reasoning fully in a few sentences, "
        f"and state which option you align with.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{formatted_options}"
    )

    # Wrap the raw text in the conversational dictionary format
    chat_constrained = [{"role": "user", "content": raw_constrained}]
    chat_unconstrained = [{"role": "user", "content": raw_unconstrained}]

    # Apply the Qwen Chat Template so the model knows to answer as an assistant
    # add_generation_prompt=True adds the final token that cues the AI to start speaking
    prompt_constrained = tokenizer.apply_chat_template(chat_constrained, tokenize=False, add_generation_prompt=True)
    prompt_unconstrained = tokenizer.apply_chat_template(chat_unconstrained, tokenize=False, add_generation_prompt=True)

    return {
        "Prompt_Constrained": prompt_constrained,
        "Prompt_Unconstrained": prompt_unconstrained
    }

prepared_dataset = dataset.map(standardize_and_prompt)
print("Data preparation complete!")
print(len(prepared_dataset))




Loading model and tokenizer...
Loading and standardizing dataset...


Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Data preparation complete!
900


In [13]:
import re
import polars as pl
from vllm import SamplingParams

# ==========================================
# 1. CONFIGURE SAMPLING PARAMETERS
# ==========================================
# FIX: Increase max_tokens to 5 to avoid the "Whitespace Trap"
params_run1 = SamplingParams(max_tokens=5, logprobs=5, temperature=0.0)

# Run 2: Unconstrained text generation up to 256 tokens
params_run2 = SamplingParams(max_tokens=256, temperature=0.7)

# ==========================================
# 2. DUAL-RUN INFERENCE LOOP
# ==========================================
print("Starting Dual-Run Inference Loop...")
results = []

print(f"Running inference..., Total number of questions is {len(prepared_dataset)}")

# FIX: Use enumerate() to automatically generate an index number for your ID
for idx, row in enumerate(prepared_dataset):
    # Create a surrogate ID using the loop index
    q_id = f"Q_{idx}"
    print(q_id)

    # Grab the dataset source (MMLU, OpinionQA, or TruthfulQA) if it exists
    dataset_source = row.get("Dataset_Source", "Unknown")

    # --- 1: First-Token Logprobs ---
    out_run1 = llm.generate([row["Prompt_Constrained"]], params_run1, use_tqdm=False)

    # FIX: Extract using Regex instead of .strip() to handle leading spaces
    run1_raw_text = out_run1[0].outputs[0].text
    match_run1 = re.search(r'\b([A-D])\b', run1_raw_text)
    first_token = match_run1.group(1) if match_run1 else "UNKNOWN"

    logprobs_dict = out_run1[0].outputs[0].logprobs[0]

    # --- 2: Unconstrained Text Generation ---
    out_run2 = llm.generate([row["Prompt_Unconstrained"]], params_run2, use_tqdm=False)
    generated_text = out_run2[0].outputs[0].text

    # Parse choice letter (A, B, C, or D) from generated text
    match_run2 = re.search(r'\b([A-D])\b', generated_text)
    text_answer = match_run2.group(1) if match_run2 else "Refusal/Unclear"

    # Calculate Mismatch
    is_mismatch = (first_token != text_answer)

    results.append({
        "question_id": q_id,
        "dataset_source": dataset_source,
        "first_token_answer": first_token,
        "unconstrained_text": generated_text,
        "parsed_text_answer": text_answer,
        "is_mismatch": is_mismatch,
        "top_1st_token_logprobs": str(logprobs_dict)
    })

# ==========================================
# 3. CREATE df_results DATAFRAME
# ==========================================
df_results = pl.DataFrame(results)

# Expand display width and print summary table
pl.Config.set_tbl_width_chars(100)
print("\n--- RESULTS SUMMARY ---")

# Updated to include your dataset source in the printout
print(df_results.select([
    "question_id",
    "dataset_source",
    "first_token_answer",
    "parsed_text_answer",
    "is_mismatch"
]))




Starting Dual-Run Inference Loop...
Running inference..., Total number of questions is 900
Q_0
Q_1
Q_2
Q_3
Q_4
Q_5
Q_6
Q_7
Q_8
Q_9
Q_10
Q_11
Q_12
Q_13
Q_14
Q_15
Q_16
Q_17
Q_18
Q_19
Q_20
Q_21
Q_22
Q_23
Q_24
Q_25
Q_26
Q_27
Q_28
Q_29
Q_30
Q_31
Q_32
Q_33
Q_34
Q_35
Q_36
Q_37
Q_38
Q_39
Q_40
Q_41
Q_42
Q_43
Q_44
Q_45
Q_46
Q_47
Q_48
Q_49
Q_50
Q_51
Q_52
Q_53
Q_54
Q_55
Q_56
Q_57
Q_58
Q_59
Q_60
Q_61
Q_62
Q_63
Q_64
Q_65
Q_66
Q_67
Q_68
Q_69
Q_70
Q_71
Q_72
Q_73
Q_74
Q_75
Q_76
Q_77
Q_78
Q_79
Q_80
Q_81
Q_82
Q_83
Q_84
Q_85
Q_86
Q_87
Q_88
Q_89
Q_90
Q_91
Q_92
Q_93
Q_94
Q_95
Q_96
Q_97
Q_98
Q_99
Q_100
Q_101
Q_102
Q_103
Q_104
Q_105
Q_106
Q_107
Q_108
Q_109
Q_110
Q_111
Q_112
Q_113
Q_114
Q_115
Q_116
Q_117
Q_118
Q_119
Q_120
Q_121
Q_122
Q_123
Q_124
Q_125
Q_126
Q_127
Q_128
Q_129
Q_130
Q_131
Q_132
Q_133
Q_134
Q_135
Q_136
Q_137
Q_138
Q_139
Q_140
Q_141
Q_142
Q_143
Q_144
Q_145
Q_146
Q_147
Q_148
Q_149
Q_150
Q_151
Q_152
Q_153
Q_154
Q_155
Q_156
Q_157
Q_158
Q_159
Q_160
Q_161
Q_162
Q_163
Q_164
Q_165
Q_166
Q_167
Q_168
Q_169

In [22]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")


Mismatch Rate: 29.33%


In [13]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")


Mismatch Rate: 25.11%


In [15]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")


Mismatch Rate: 33.89%


In [14]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")


Mismatch Rate: 49.33%


# Saving the llm output to csv

In [15]:
#import os
#from google.colab import drive
#from google.colab import files


# 1. Mount Google Drive to this Colab session
#drive.mount('/content/drive')

safe_model_name = selected_option.split("/")[0]

# 2. Define the Google Drive paths
# You can change 'MyDrive' to a specific folder path if you want (e.g., 'MyDrive/Colab Notebooks/experiment_results.csv')
csv_path = f'/content/drive/MyDrive/Colab Notebooks/NLP Poster/{safe_model_name}_dataset_experiment_results.csv'


# 1. Save the Polars DataFrame to a CSV file
df_results.write_csv(csv_path)

# The feture extraction -- Elfen part

In [1]:
!sudo apt-get update -qq
!sudo apt-get install -y python3.12 python3.12-venv python3.12-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpython3.12 libpython3.12-dev libpython3.12-stdlib
The following NEW packages will be installed:
  libpython3.12 libpython3.12-dev libpython3.12-stdlib python3.12
  python3.12-dev python3.12-venv
0 upgraded, 6 newly installed, 0 to remove and 11 not upgraded.
Need to get 15.7 MB of archives.
After this operation, 62.7 MB of additional disk space will be used.
Get:1 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.12-stdlib amd64 3.12.13-1+jammy1 [2,866 kB]
Get:2 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main amd64 libpython3.12 amd64 3.12.13-1+jammy1 [2,375 kB]
Get:3 https://ppa.launchpadcontent.n

In [2]:
!python3.12 -m venv /content/elfen_venv
!/content/elfen_venv/bin/pip install --upgrade pip
!/content/elfen_venv/bin/pip install elfen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.0.1
    Uninstalling pip-25.0.1:
      Successfully uninstalled pip-25.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.2/33.2 MB 41.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 65.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 115.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 990.1/990.1 kB 33.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 32.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.0/865.0 kB 20.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━

In [12]:
!/content/elfen_venv/bin/pip install click
!/content/elfen_venv/bin/python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 21.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [14]:
%%script /content/elfen_venv/bin/python
import polars as pl
import elfen
from elfen import Extractor

# ----------------------------------------------------
# A. Read CSV file
# ----------------------------------------------------
HF_csv = pl.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP Poster/HuggingFaceTB_dataset_experiment_results.csv")
QWEN_csv = pl.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP Poster/QWEN_dataset_experiment_results.csv")
google_csv = pl.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP Poster/google_dataset_experiment_results.csv")
meta_csv = pl.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP Poster/meta-llama_dataset_experiment_results.csv")
mistral_csv = pl.read_csv("/content/drive/MyDrive/Colab Notebooks/NLP Poster/mistralai_dataset_experiment_results.csv")
print(f"the shape of the database is: {len(HF_csv)}")
print("=== Loaded CSV DataFrame ===")
print(HF_csv.head())


# ----------------------------------------------------
# B. initializing Elfen
# ----------------------------------------------------
print("\nInitializing elfen feature extractor...")
extractor = Extractor(
    HF_csv,
    language="en",
    backbone="spacy",
    text_column="unconstrained_text"
)

print("Extracting 1,061 linguistic features...")
extractor.extract_features()
df_features = extractor.get_data()



Process is terminated.


  Using cached elfen-1.0.6-py3-none-any.whl.metadata (2.6 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user


In [10]:
%%script /content/elfen_venv/bin/python

from elfen import Extractor

print("\nInitializing elfen feature extractor...")
extractor = Extractor(
    HF_csv,
    language="en",
    backbone="spacy",
    text_column="unconstrained_text"
)

print("Extracting 1,061 linguistic features...")
extractor.extract_features()
df_features = extractor.get_data()


Initializing elfen feature extractor...


/content/elfen_venv/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Traceback (most recent call last):
  File "<stdin>", line 6, in <module>
NameError: name 'HF_csv' is not defined


CalledProcessError: Command 'b'\nfrom elfen import Extractor\n\nprint("\\nInitializing elfen feature extractor...")\nextractor = Extractor(\n    HF_csv,\n    language="en",\n    backbone="spacy",\n    text_column="unconstrained_text"\n)\n\nprint("Extracting 1,061 linguistic features...")\nextractor.extract_features()\ndf_features = extractor.get_data()\n'' returned non-zero exit status 1.